In [ ]:
import numpy as np 
import os
import sys
import glob
from dscribe.descriptors import SOAP #Recommend using a new environment through conda forge 

In [ ]:
import re
#Functions needed to extract coordinates from CREST workflow
def locateinLog(logFile, textStr, returnType: str):
    matchingLines = []

    with open(logFile, "r") as f:
        for idx, line in enumerate(f):
            if textStr in line:
                matchingLines.append(idx)
    if len(matchingLines) != 0:
        if returnType == "earliest":
            return matchingLines[0]

        elif returnType == "latest":
            return matchingLines[-1]
        else:
            indx = int(input(f"Enter the index of preference for {textStr}"))
            return matchingLines[indx]
    else:
        print(logFile)
        print("Bad Log File")
        return "Poison"
def getAtomCoords(logFile , xyzStr , commaSplit:int , ):
    #Extracts atom coordinates into a dict from a log file
    atomCoords = {}
    lowerIdx = locateinLog(logFile , xyzStr, "latest" )
    upperIdx = locateinLog(logFile, "The archive entry for this job was punched." , "latest")
    if not "Poison" in [lowerIdx , upperIdx]:
        masterStr = ""
        with open(logFile , "r") as f:
            for idx, line in enumerate(f):
                if idx >= lowerIdx and idx < upperIdx:
                    cleaned = re.sub(r'\s+', '' , line)
                    masterStr += cleaned
        masterList = masterStr.split("\\")
        for i ,  phrase in enumerate(masterList):
            atomStr = phrase.split(",")
            #print(atomStr)
            if len(atomStr) == commaSplit:
                atomCoords[i] = atomStr[:commaSplit]
        return atomCoords
    else:
        return "Poison"

In [ ]:
delta50_5_0 = '/home/danny/Downloads/delta50Rep/alkenes/Crest/nmrDelta50_05_conf_0.log' #replace with the example file if your choosing
delta50_29_0 = '/home/danny/Downloads/delta50Rep/alkenes/Crest/nmrDelta50_29_conf_0.log'

In [ ]:
delta50_5_Coords = getAtomCoords(delta50_5_0 , "GINC-COMPUTE" , 5) #start by extracting the optimized coordinates from your output file 
delta50_29_Coords = getAtomCoords(delta50_29_0 , "GINC-COMPUTE" , 5)
for key , val in delta50_5_Coords.items():
    print(key, val)

In [ ]:
from ase import Atoms
#Converts atomic coordinate hash to a list of symbols and positions for ase to read 
def atomHash_to_AseForm(atomHash):
    symbols = []
    coordinates = []
    for atom , coord in atomHash.items():
        atom = str(coord[0])
        symbols.append(atom)
        coords = [float(atom) for atom in coord[2:5]]
        coordinates.append(coords)
    return symbols , np.array(coordinates)
symbols , positions = atomHash_to_AseForm(delta50_5_Coords)

mol = Atoms(symbols=symbols, positions=positions)




In [ ]:
species = list(set(symbols))
print(symbols)
r_cut = 3.0 #radial distance in Angstroms
n_max = 7 #number of radial basis functions
l_max = 8 #max degrees of spherical harmonics
soap = SOAP(
    species=species,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
)

In [ ]:
import sparse
soapParameters = soap.create(mol , centers = [2,3])


In [ ]:
delta50_59_Coords = getAtomCoords(delta50_5_0, "GINC-COMPUTE" , 5)
symbols , positions = atomHash_to_AseForm(delta50_59_Coords)
print(symbols)
print(positions)
mol = Atoms(symbols=symbols, positions=positions)
from ase.visualize import view
view(mol)